# Graphs

## 1. Overview

A graph G = (V, E) consists of a finite set of vertices V and a set of edges E connecting pairs of vertices.

- Social networks: vertices = people, edges = friendships
- Road maps: vertices = intersections, edges = roads
- Dependencies: vertices = tasks, edges = prerequisites
- State machines: vertices = states, edges = transitions

| Classification | Types | Distinction |
|---|---|---|
| Direction | Undirected / Directed | Edges have no direction vs. edges are ordered pairs |
| Weight | Unweighted / Weighted | Edges carry no cost vs. edges have numeric weights |
| Cycles | Cyclic / Acyclic | Contains at least one cycle vs. no cycles (DAG if directed) |
| Connectivity | Connected / Disconnected | Path exists between all vertex pairs vs. isolated components |
| Density | Dense / Sparse | E close to V^2 vs. E much less than V^2 |

### Key Terms

- **Vertex (node):** basic unit of a graph
- **Edge:** connection between two vertices; undirected = (u, v), directed = <u, v>
- **Degree:** number of edges incident to a vertex
  - Undirected: total edges connected
  - Directed: **in-degree** (edges arriving) + **out-degree** (edges leaving)
- **Path:** sequence of vertices where each adjacent pair is connected by an edge
- **Cycle:** path whose start and end vertex coincide
- **Simple path:** no vertex appears twice (except start/end in a cycle)
- **DAG:** directed acyclic graph, used in topological sort, DP, scheduling
- **Connected component:** maximal subgraph where all vertices are mutually reachable (undirected)
- **Strongly connected component (SCC):** maximal subgraph where all vertices are mutually reachable via directed paths
- **Complete graph:** every pair of vertices is connected; V*(V-1)/2 edges (undirected), V*(V-1) edges (directed)

## 2. Graph Representations

Two primary storage methods: adjacency matrix and adjacency list.

| Operation | Adjacency Matrix | Adjacency List (hash table) |
|---|---|---|
| **Space** | O(V^2) | O(V + E) |
| **Check edge (u, v)** | O(1) | O(1) avg (hash) |
| **All neighbors of u** | O(V) | O(degree(u)) |
| **Add edge** | O(1) | O(1) |
| **Build graph** | O(V^2) | O(V + E) |
| **Best for** | Dense graphs, edge existence queries | Sparse graphs, neighbor traversal |

Adjacency lists are the default for most graph problems. A hash-table adjacency list maps each vertex to a dict of `{neighbor: weight}`.

### Adjacency List (Hash Table)

Standard construction for interview problems. Each vertex maps to a dict of `{neighbor: weight}`.

```text
Graph:  1 --5-- 2 --7-- 4
        |               |
        6               9
        |               |
        5 --8-- 6 --3-- 3 --2-- 1

Adjacency list (directed):
1: {2: 5, 5: 6}
2: {4: 7}
3: {1: 2}
4: {3: 9}
5: {6: 8}
6: {4: 3}
```

In [ ]:
def buildGraph(n, edges, directed=True):
    graph = {i: {} for i in range(1, n + 1)}
    for u, v, w in edges:
        graph[u][v] = w
        if not directed:
            graph[v][u] = w
    return graph


edges = [(1, 2, 5), (1, 5, 6), (2, 4, 7), (4, 3, 9), (3, 1, 2), (5, 6, 8), (6, 4, 3)]
graph = buildGraph(6, edges)

graph[4].get(3, None)  # 9
graph[4].get(5, None)  # None

### Adjacency Matrix

Store edge weights in a V x V 2D array. `adj[i][j] = w` if edge exists, `inf` otherwise. Diagonal is 0.

Best for dense graphs or when O(1) edge existence checks dominate.

In [ ]:
def buildMatrix(n, edges, directed=True):
    INF = float("inf")
    adj = [[INF] * (n + 1) for _ in range(n + 1)]
    for i in range(n + 1):
        adj[i][i] = 0
    for u, v, w in edges:
        adj[u][v] = w
        if not directed:
            adj[v][u] = w
    return adj

## 3. Depth-First Search (DFS)

Explore as deep as possible along each branch before backtracking. Uses recursion (implicit stack) or an explicit stack. Visits each vertex and edge once.

| Property | Detail |
|---|---|
| **Key idea** | Go deep along one path, backtrack when stuck |
| **Use-case** | Connected components, cycle detection, path finding: `graph = {1: [2,3], 2: [4], ...}` |
| **Problem keywords** | "number of islands", "connected components", "detect cycle", "all paths" |
| **Time** | O(V + E) |
| **Space** | O(V) for visited set + recursion stack |

### Pseudocode

$$
\begin{array}{l}
\hline
\textbf{Pattern: } \text{depthFirstSearch} \\
\hline
\textbf{function } \text{dfs}(graph, u, visited): \\
\quad 1: \quad \text{mark } u \text{ as visited} \\
\quad 2: \quad \textbf{for each } \text{unvisited neighbor } v \text{ of } u: \\
\quad 3: \quad\quad \text{recurse into } v \text{ before trying the next neighbor} \\
\quad 4: \quad \text{all of } u\text{'s neighbors explored, return to caller (backtrack)} \\
\hline
\end{array}
$$

In [2]:
def dfs(graph, u, visited):
    visited.add(u)
    for v in graph[u]:
        if v not in visited:
            dfs(graph, v, visited)

#### Steps

Graph adjacency list: `1:[2,3]  2:[1,3,4]  3:[1,2,4,5]  4:[2,3,5,6]  5:[3,4]  6:[4,7]  7:[]`

DFS from vertex 1:

| Step | Current | Action | Stack (implicit) | Visited |
|------|---------|--------|-------------------|---------|
| 1 | 1 | visit, recurse into 2 | [1] | {1} |
| 2 | 2 | visit, recurse into 3 (1 visited) | [1, 2] | {1, 2} |
| 3 | 3 | visit, recurse into 4 (1,2 visited) | [1, 2, 3] | {1, 2, 3} |
| 4 | 4 | visit, recurse into 5 (2,3 visited) | [1, 2, 3, 4] | {1, 2, 3, 4} |
| 5 | 5 | visit, 3 and 4 visited, backtrack | [1, 2, 3, 4] | {1, 2, 3, 4, 5} |
| 6 | 4 | resume, recurse into 6 | [1, 2, 3, 4] | {1, 2, 3, 4, 5} |
| 7 | 6 | visit, recurse into 7 | [1, 2, 3, 4, 6] | {1, 2, 3, 4, 5, 6} |
| 8 | 7 | visit, no neighbors, backtrack all | [1, 2, 3, 4, 6, 7] | {1, 2, 3, 4, 5, 6, 7} |

Visit order: 1, 2, 3, 4, 5, 6, 7

### Application: Number of Islands (LC 200)

Treat a 2D grid as a graph. Each `'1'` cell connects to its 4-directional neighbors. DFS from each unvisited `'1'`, mark all reachable land as visited. Count the number of DFS calls.

In [ ]:
def numIslands(grid):
    rows, cols = len(grid), len(grid[0])

    def sink(i, j):
        if i < 0 or i >= rows or j < 0 or j >= cols or grid[i][j] == "0":
            return
        grid[i][j] = "0"
        sink(i + 1, j)
        sink(i - 1, j)
        sink(i, j + 1)
        sink(i, j - 1)

    cnt = 0
    for i in range(rows):
        for j in range(cols):
            if grid[i][j] == "1":
                sink(i, j)
                cnt += 1
    return cnt

## 4. Breadth-First Search (BFS)

Expand outward level by level from the starting vertex. Visits all vertices at distance d before any vertex at distance d+1. Uses a queue.

| Property | Detail |
|---|---|
| **Key idea** | Explore neighbors level by level using a queue |
| **Use-case** | Shortest path in unweighted graphs, level-order: `graph = {1: [2,3], ...}` |
| **Problem keywords** | "shortest path unweighted", "minimum steps", "level order", "word ladder" |
| **Time** | O(V + E) |
| **Space** | O(V) for visited set + queue |

### Pseudocode

$$
\begin{array}{l}
\hline
\textbf{Pattern: } \text{breadthFirstSearch} \\
\hline
\textbf{function } \text{bfs}(graph, source): \\
\quad 1: \quad visited \leftarrow \{source\}, \; queue \leftarrow [source] \\
\quad 2: \quad \textbf{while } queue \text{ not empty}: \\
\quad 3: \quad\quad u \leftarrow queue.\text{popleft}() \\
\quad 4: \quad\quad \textbf{for each } \text{neighbor } v \text{ of } u: \\
\quad 5: \quad\quad\quad \textbf{if } v \notin visited: \\
\quad 6: \quad\quad\quad\quad \text{mark } v \text{ visited and enqueue } v \\
\hline
\end{array}
$$

Mark visited **at enqueue time** (step 6), not at dequeue (step 3). If marking happens at dequeue, multiple copies of the same vertex can sit in the queue simultaneously.

In [ ]:
from collections import deque


def bfs(graph, source):
    visited = {source}
    queue = deque([source])
    order = []
    while queue:
        u = queue.popleft()
        order.append(u)
        for v in graph[u]:
            if v not in visited:
                visited.add(v)
                queue.append(v)
    return order

#### Steps

Same graph. BFS from vertex 1:

| Step | Dequeue | Enqueue | Queue state | Visited |
|------|---------|---------|-------------|---------|
| 1 | 1 | 2, 3 | [2, 3] | {1, 2, 3} |
| 2 | 2 | 4 | [3, 4] | {1, 2, 3, 4} |
| 3 | 3 | 5 | [4, 5] | {1, 2, 3, 4, 5} |
| 4 | 4 | 6 | [5, 6] | {1, 2, 3, 4, 5, 6} |
| 5 | 5 | -- | [6] | {1, 2, 3, 4, 5, 6} |
| 6 | 6 | 7 | [7] | {1, 2, 3, 4, 5, 6, 7} |
| 7 | 7 | -- | [] | {1, 2, 3, 4, 5, 6, 7} |

Visit order: 1, 2, 3, 4, 5, 6, 7

**DFS vs BFS:** DFS goes deep first (stack/recursion), BFS goes wide first (queue). BFS finds shortest path in unweighted graphs. DFS is simpler for exhaustive exploration or backtracking.

## 5. Topological Sort

Order the vertices of a DAG into a line so every edge points left to right.

- If edge <u, v> exists, u must appear before v in the output
- Only works on directed acyclic graphs
- If a cycle exists, no valid ordering is possible
- A DAG can have multiple valid topological orders

| Property | Detail |
|---|---|
| **Key idea** | Order vertices so all edges point forward |
| **Use-case** | Task scheduling, course prerequisites: `[[1,0], [2,0], [3,1], [3,2]]` |
| **Problem keywords** | "course schedule", "task order", "dependency", "DAG ordering" |
| **Time** | O(V + E) |
| **Space** | O(V + E) |
| **Cycle detection** | If result length < V, a cycle exists |

### Kahn's Algorithm (BFS-based)

Process vertices that have no remaining prerequisites (in-degree = 0).

- Start by enqueuing all vertices with in-degree 0
- Each time a vertex is removed from the queue, it is added to the output
- Removing a vertex decrements its neighbors' in-degrees
- When a neighbor's in-degree drops to 0, it becomes eligible and enters the queue
- If the queue empties before all vertices are output, a cycle exists

$$
\begin{array}{l}
\hline
\textbf{Pattern: } \text{kahnTopologicalSort} \\
\hline
\textbf{function } \text{topoSortKahn}(graph): \\
\quad 1: \quad \text{count in-degree for every vertex} \\
\quad 2: \quad \text{enqueue all vertices with in-degree } 0 \\[4pt]
\quad 3: \quad \textbf{while } queue \text{ not empty}: \\
\quad 4: \quad\quad u \leftarrow queue.\text{popleft}() \\
\quad 5: \quad\quad \text{append } u \text{ to } order \\
\quad 6: \quad\quad \textbf{for each } \text{neighbor } v \text{ of } u: \\
\quad 7: \quad\quad\quad \text{decrement indegree}[v] \\
\quad 8: \quad\quad\quad \textbf{if } \text{indegree}[v] = 0: \text{ enqueue } v \\[4pt]
\quad 9: \quad \textbf{if } |order| < |V|: \text{ cycle exists} \\
\hline
\end{array}
$$

In [ ]:
def topoSortKahn(graph):
    indegrees = {u: 0 for u in graph}
    for u in graph:
        for v in graph[u]:
            indegrees[v] += 1

    queue = deque([u for u in indegrees if indegrees[u] == 0])
    order = []

    while queue:
        u = queue.popleft()
        order.append(u)
        for v in graph[u]:
            indegrees[v] -= 1
            if indegrees[v] == 0:
                queue.append(v)

    return order if len(order) == len(graph) else []

#### Steps

```text
DAG: 0 -> 1 -> 3
     0 -> 2 -> 3
```

| Step | Dequeue | Decrement in-degrees | Queue | Order |
|------|---------|---------------------|-------|-------|
| init | -- | indeg: {0:0, 1:1, 2:1, 3:2} | [0] | [] |
| 1 | 0 | 1: 1->0, 2: 1->0 | [1, 2] | [0] |
| 2 | 1 | 3: 2->1 | [2] | [0, 1] |
| 3 | 2 | 3: 1->0 | [3] | [0, 1, 2] |
| 4 | 3 | -- | [] | [0, 1, 2, 3] |

Result: `[0, 1, 2, 3]` (valid topological order)

### DFS-based Topological Sort

Finish a vertex only after all its descendants are finished. Append each vertex to the result after its recursive calls return, then reverse.

- Visit a vertex, recurse into all unvisited neighbors
- After all neighbors are fully processed, append the current vertex
- Reverse the collected list at the end to get topological order
- Track the current recursion path with `onStack`; if a vertex is revisited while still on the path, a cycle exists

In [ ]:
def topoSortDFS(graph):
    visited = set()
    onStack = set()
    order = []
    hasCycle = False

    def dfs(u):
        nonlocal hasCycle
        if hasCycle or u in onStack:
            hasCycle = True
            return
        if u in visited:
            return
        visited.add(u)
        onStack.add(u)
        for v in graph[u]:
            dfs(v)
        order.append(u)
        onStack.remove(u)

    for u in graph:
        if u not in visited:
            dfs(u)

    if hasCycle:
        return []
    order.reverse()
    return order

## 6. Minimum Spanning Tree

A spanning tree of a connected undirected graph contains all V vertices and exactly V-1 edges with no cycles. The minimum spanning tree (MST) is the spanning tree with the smallest total edge weight.

| Algorithm | Strategy | Time | Space | Best for |
|---|---|---|---|---|
| Prim | Grow tree from a vertex, add cheapest crossing edge | O(V^2) or O((V+E) log V) with heap | O(V) | Dense graphs |
| Kruskal | Sort edges, add cheapest edge that does not create a cycle | O(E log E) | O(V) | Sparse graphs |

### Prim's Algorithm

Start from any vertex. Repeatedly pick the cheapest edge that connects a vertex inside the MST to a vertex outside it. Add that vertex and edge to the MST.

- `cost[v]` = cheapest single edge from any MST vertex to v (not a cumulative path length)
- Each round: find the unvisited vertex with the smallest `cost`, mark it visited, add `cost[v]` to the total
- Then update `cost` for its unvisited neighbors

| Property | Detail |
|---|---|
| **Key idea** | Grow MST one vertex at a time via the cheapest crossing edge |
| **Use-case** | Min cost to connect all points: `points = [[0,0],[2,2],[3,10],[5,2],[7,0]]` |
| **Problem keywords** | "minimum spanning tree", "connect all nodes", "minimum cost" |
| **Time** | O(V^2) naive, O((V+E) log V) with priority queue |
| **Space** | O(V) |

$$
\begin{array}{l}
\hline
\textbf{Pattern: } \text{prim} \\
\hline
\textbf{function } \text{prim}(graph, start): \\
\quad 1: \quad cost[start] \leftarrow 0, \; \text{all others} \leftarrow \infty \\
\quad 2: \quad \textbf{repeat } V \text{ times}: \\
\quad 3: \quad\quad u \leftarrow \text{unvisited vertex with smallest } cost \\
\quad 4: \quad\quad \text{mark } u \text{ visited, add } cost[u] \text{ to total} \\
\quad 5: \quad\quad \textbf{for each } \text{unvisited neighbor } v \text{ of } u: \\
\quad 6: \quad\quad\quad \textbf{if } w(u,v) < cost[v]: \\
\quad 7: \quad\quad\quad\quad cost[v] \leftarrow w(u,v) \\
\hline
\end{array}
$$

In [ ]:
def prim(graph, n, start):
    vis = set()
    dist = [float("inf")] * n
    dist[start] = 0
    ans = 0

    for _ in range(n):
        u = -1
        for i in range(n):
            if i not in vis and (u == -1 or dist[i] < dist[u]):
                u = i
        if dist[u] == float("inf"):
            return -1
        vis.add(u)
        ans += dist[u]
        for v, w in graph.get(u, {}).items():
            if v not in vis and w < dist[v]:
                dist[v] = w
    return ans

### Kruskal's Algorithm

Greedy: sort all edges by weight. Process edges lightest first. Add an edge only if its two endpoints are in different components (no cycle). Use union-find to track components.

| Property | Detail |
|---|---|
| **Key idea** | Sort edges ascending, greedily add if no cycle forms |
| **Use-case** | Same MST problems, especially when edges are given as a list |
| **Problem keywords** | "minimum spanning tree", "min cost to connect" |
| **Time** | O(E log E) dominated by sorting |
| **Space** | O(V) for union-find |

$$
\begin{array}{l}
\hline
\textbf{Pattern: } \text{kruskal} \\
\hline
\textbf{function } \text{kruskal}(edges, V): \\
\quad 1: \quad \text{sort } edges \text{ by weight ascending} \\
\quad 2: \quad \text{initialize union-find with } V \text{ components} \\[4pt]
\quad 3: \quad \textbf{for each } (u, v, w) \in edges: \\
\quad 4: \quad\quad \textbf{if } u \text{ and } v \text{ are in different components}: \\
\quad 5: \quad\quad\quad \text{merge their components, add } w \text{ to total} \\
\quad 6: \quad\quad\quad \textbf{if } V - 1 \text{ edges added: done} \\
\hline
\end{array}
$$

In [ ]:
class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))

    def find(self, x):
        while x != self.parent[x]:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, x, y):
        rx, ry = self.find(x), self.find(y)
        if rx == ry:
            return False
        self.parent[rx] = ry
        return True


def kruskal(edges, n):
    uf = UnionFind(n)
    edges.sort(key=lambda x: x[2])
    ans = 0
    cnt = 0
    for u, v, w in edges:
        if uf.union(u, v):
            ans += w
            cnt += 1
            if cnt == n - 1:
                break
    return ans

## 7. Single-Source Shortest Path

Find the shortest path from one source vertex to all other vertices in a weighted graph.

| Algorithm | Time | Space | Negative edges | Negative cycles |
|---|---|---|---|---|
| Dijkstra (heap) | O((V+E) log V) | O(V) | No | No |
| Bellman-Ford | O(VE) | O(V) | Yes | Detects |

### Dijkstra's Algorithm (Heap-Optimized)

Greedy: always process the unvisited vertex with the smallest known distance. Use a min-heap to select it efficiently. For each neighbor, check if routing through the current vertex is cheaper (relaxation). Cannot handle negative edge weights.

| Property | Detail |
|---|---|
| **Key idea** | Lock the nearest unlocked vertex, relax its neighbors |
| **Use-case** | Weighted graph shortest path: `graph = {1: {2:2, 3:4}, 2: {3:1, 4:7}, ...}` |
| **Problem keywords** | "shortest path", "minimum cost path", "network delay" |
| **Time** | O((V+E) log V) |
| **Space** | O(V) |
| **Constraint** | All edge weights must be non-negative |

$$
\begin{array}{l}
\hline
\textbf{Pattern: } \text{dijkstra} \\
\hline
\textbf{function } \text{dijkstra}(graph, n, source): \\
\quad 1: \quad dist[source] \leftarrow 0, \; \text{all others} \leftarrow \infty \\
\quad 2: \quad \text{push } (0, source) \text{ onto min-heap} \\[4pt]
\quad 3: \quad \textbf{while } heap \text{ not empty}: \\
\quad 4: \quad\quad (d, u) \leftarrow \text{pop smallest from heap} \\
\quad 5: \quad\quad \textbf{if } d > dist[u]: \text{ skip (stale entry)} \\
\quad 6: \quad\quad \textbf{for each } \text{neighbor } (v, w) \text{ of } u: \\
\quad 7: \quad\quad\quad \textbf{if } dist[u] + w < dist[v]: \\
\quad 8: \quad\quad\quad\quad dist[v] \leftarrow dist[u] + w \\
\quad 9: \quad\quad\quad\quad \text{push } (dist[v], v) \text{ onto heap} \\
\hline
\end{array}
$$

In [ ]:
import heapq


def dijkstra(graph, n, source):
    dist = [float("inf")] * (n + 1)
    dist[source] = 0
    heap = [(0, source)]

    while heap:
        d, u = heapq.heappop(heap)
        if d > dist[u]:
            continue
        for v, w in graph.get(u, {}).items():
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
                heapq.heappush(heap, (dist[v], v))
    return dist

#### Steps

```text
graph: 1 --(2)--> 2 --(1)--> 3 --(3)--> 4
       1 --(4)--> 3          2 --(7)--> 4
```

Dijkstra from source = 1:

| Step | Pop (d, u) | Relax | dist updates | Heap |
|------|-----------|-------|-------------|------|
| init | -- | -- | dist = [-, 0, inf, inf, inf] | [(0,1)] |
| 1 | (0, 1) | 1->2: 0+2=2, 1->3: 0+4=4 | dist[2]=2, dist[3]=4 | [(2,2), (4,3)] |
| 2 | (2, 2) | 2->3: 2+1=3 < 4, 2->4: 2+7=9 | dist[3]=3, dist[4]=9 | [(3,3), (4,3), (9,4)] |
| 3 | (3, 3) | 3->4: 3+3=6 < 9 | dist[4]=6 | [(4,3), (9,4), (6,4)] |
| 4 | (4, 3) | stale (3 > dist[3]=3), skip | -- | [(6,4), (9,4)] |
| 5 | (6, 4) | no neighbors | -- | [(9,4)] |
| 6 | (9, 4) | stale, skip | -- | [] |

Result: `dist = [-, 0, 2, 3, 6]`

### Bellman-Ford Algorithm

Relaxation-based: repeat V-1 rounds of relaxing every edge. After V-1 rounds, all shortest paths are finalized. A Vth round that still finds improvements proves a negative-weight cycle exists.

| Property | Detail |
|---|---|
| **Key idea** | Relax all edges V-1 times; a Vth relaxation detects negative cycles |
| **Use-case** | Graphs with negative edge weights, arbitrage detection |
| **Problem keywords** | "negative weights", "cheapest flights with k stops" |
| **Time** | O(VE) |
| **Space** | O(V) |

$$
\begin{array}{l}
\hline
\textbf{Pattern: } \text{bellmanFord} \\
\hline
\textbf{function } \text{bellmanFord}(graph, n, source): \\
\quad 1: \quad dist[source] \leftarrow 0, \; \text{all others} \leftarrow \infty \\[4pt]
\quad 2: \quad \textbf{repeat } V - 1 \text{ times}: \\
\quad 3: \quad\quad \textbf{for each edge } (u, v, w): \\
\quad 4: \quad\quad\quad \textbf{if } dist[u] + w < dist[v]: \; dist[v] \leftarrow dist[u] + w \\[4pt]
\quad 5: \quad \textbf{for each edge } (u, v, w): \\
\quad 6: \quad\quad \textbf{if } dist[u] + w < dist[v]: \text{ negative cycle exists} \\
\hline
\end{array}
$$

In [ ]:
def bellmanFord(graph, n, source):
    dist = [float("inf")] * (n + 1)
    dist[source] = 0

    for _ in range(n - 1):
        updated = False
        for u in graph:
            for v, w in graph[u].items():
                if dist[u] != float("inf") and dist[u] + w < dist[v]:
                    dist[v] = dist[u] + w
                    updated = True
        if not updated:
            break

    for u in graph:
        for v, w in graph[u].items():
            if dist[u] != float("inf") and dist[u] + w < dist[v]:
                return None
    return dist

## 8. Multi-Source Shortest Path

### Floyd-Warshall Algorithm

Dynamic programming over intermediate vertices. For each vertex k, check whether routing i -> k -> j is shorter than the current i -> j. Three nested loops: outer over intermediates, inner over all pairs.

| Property | Detail |
|---|---|
| **Key idea** | Try every vertex as a waypoint between every pair |
| **Use-case** | All-pairs shortest path, small dense graphs |
| **Problem keywords** | "all pairs shortest path", "distance matrix", "transitive closure" |
| **Time** | O(V^3) |
| **Space** | O(V^2) |
| **Constraint** | No negative cycles. Detect via `dist[i][i] < 0` |

$$
\begin{array}{l}
\hline
\textbf{Pattern: } \text{floydWarshall} \\
\hline
\textbf{function } \text{floydWarshall}(graph, n): \\
\quad 1: \quad \text{initialize } dist[i][j] \text{ from edge weights; } \infty \text{ if no edge; } 0 \text{ on diagonal} \\[4pt]
\quad 2: \quad \textbf{for each } \text{waypoint } k: \\
\quad 3: \quad\quad \textbf{for each } \text{pair } (i, j): \\
\quad 4: \quad\quad\quad dist[i][j] \leftarrow \min(dist[i][j], \; dist[i][k] + dist[k][j]) \\
\hline
\end{array}
$$

In [ ]:
def floydWarshall(graph, n):
    INF = float("inf")
    dist = [[INF] * n for _ in range(n)]
    for i in range(n):
        dist[i][i] = 0
        for j, w in graph.get(i, {}).items():
            dist[i][j] = w

    for k in range(n):
        for i in range(n):
            if dist[i][k] == INF:
                continue
            for j in range(n):
                if dist[k][j] == INF:
                    continue
                if dist[i][j] > dist[i][k] + dist[k][j]:
                    dist[i][j] = dist[i][k] + dist[k][j]
    return dist

## 9. Bipartite Graphs

Vertices split into two groups where every edge crosses between groups.

- No edges connect vertices within the same group
- A graph is bipartite if and only if it has no odd-length cycles
- Detection: try 2-coloring the graph; if any edge has same-colored endpoints, not bipartite

### Detection (Coloring Method)

Alternate two colors via DFS. If a neighbor already has the same color as the current vertex, the graph is not bipartite.

| Property | Detail |
|---|---|
| **Key idea** | 2-color the graph; same-color neighbor = not bipartite |
| **Use-case** | Check if graph is bipartite: `graph = [[1,3],[0,2],[1,3],[0,2]]` |
| **Problem keywords** | "is graph bipartite", "two-coloring", "odd cycle" |
| **Time** | O(V + E) |
| **Space** | O(V) |

In [ ]:
def isBipartite(graph):
    n = len(graph)
    colors = [0] * n

    def colorDfs(node, color):
        colors[node] = color
        for nei in graph[node]:
            if colors[nei] == color:
                return False
            if colors[nei] == 0 and not colorDfs(nei, -color):
                return False
        return True

    for i in range(n):
        if colors[i] == 0 and not colorDfs(i, 1):
            return False
    return True

### Maximum Matching (Hungarian Algorithm)

Find the largest set of edges where no two edges share a vertex.

- For each unmatched left vertex, DFS for an **augmenting path**: a path that alternates between unmatched and matched edges, ending at an unmatched right vertex
- When found, flip all edges along the path (matched becomes unmatched and vice versa), increasing total matches by 1
- Repeat until no more augmenting paths exist

| Property | Detail |
|---|---|
| **Key idea** | For each unmatched left vertex, DFS for an augmenting path and flip it |
| **Use-case** | Job assignment, pairing problems |
| **Problem keywords** | "maximum matching", "assign tasks", "pairing" |
| **Time** | O(VE) |
| **Space** | O(V) |

In [ ]:
def maxMatching(graph, leftSize, rightSize):
    matchRight = [-1] * rightSize

    def augment(left, vis):
        for right in graph[left]:
            if not vis[right]:
                vis[right] = True
                if matchRight[right] == -1 or augment(matchRight[right], vis):
                    matchRight[right] = left
                    return True
        return False

    res = 0
    for left in range(leftSize):
        vis = [False] * rightSize
        if augment(left, vis):
            res += 1
    return res

## 10. Reference

### Algorithm Comparison

| Algorithm | Type | Time | Space | Key constraint |
|---|---|---|---|---|
| DFS | Traversal | O(V + E) | O(V) | -- |
| BFS | Traversal | O(V + E) | O(V) | Shortest path in unweighted |
| Kahn (topo sort) | Ordering | O(V + E) | O(V + E) | DAG only |
| DFS topo sort | Ordering | O(V + E) | O(V) | DAG only |
| Prim | MST | O(V^2) / O((V+E) log V) | O(V) | Undirected, connected |
| Kruskal | MST | O(E log E) | O(V) | Undirected, connected |
| Dijkstra (heap) | SSSP | O((V+E) log V) | O(V) | Non-negative weights |
| Bellman-Ford | SSSP | O(VE) | O(V) | Handles negative weights |
| Floyd-Warshall | APSP | O(V^3) | O(V^2) | No negative cycles |

### Representation Comparison

| Representation | Space | Edge check | Neighbors | Best for |
|---|---|---|---|---|
| Adjacency matrix | O(V^2) | O(1) | O(V) | Dense, Floyd-Warshall |
| Adjacency list (hash) | O(V + E) | O(1) avg | O(degree) | Most problems |

### Problem Keyword Mapping

| Keywords | Algorithm |
|---|---|
| "number of islands", "connected components", "flood fill" | DFS/BFS |
| "shortest path unweighted", "minimum steps", "word ladder" | BFS |
| "course schedule", "task dependencies", "build order" | Topological sort |
| "minimum cost to connect all", "MST" | Prim / Kruskal |
| "shortest path weighted", "network delay", "cheapest path" | Dijkstra |
| "negative weights", "cheapest flights with k stops" | Bellman-Ford |
| "all pairs shortest path", "distance matrix" | Floyd-Warshall |
| "is bipartite", "two-coloring" | DFS coloring |
| "maximum matching", "job assignment" | Hungarian |